In [ ]:
# V-JEPA 2.1 (nurikw3, 5 folds) on the 58 gold + 231 ref-B train studies. Eval only, no submission.
import glob, sys, time, json
from pathlib import Path
import numpy as np, pandas as pd, torch
CK = Path(glob.glob('/kaggle/input/**/f0_best.pt', recursive=True)[0]).parent
DATA = Path(glob.glob('/kaggle/input/**/train_series.csv', recursive=True)[0]).parent
print(CK, DATA, flush=True)
sys.path.insert(0, str(CK)); sys.path.insert(0, str(CK / 'vjepa2'))
import infer as V
REFB = ["1.2.826.0.1.3680043.8.498.10574706940531650593705137704922433109", "1.2.826.0.1.3680043.8.498.42287188181305579531212818123200473947", "1.2.826.0.1.3680043.8.498.36193046155628980489031133466840036976", "1.2.826.0.1.3680043.8.498.13291943370279119939499960846843879156", "1.2.826.0.1.3680043.8.498.12450858400030959723767712312663176092", "1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990", "1.2.826.0.1.3680043.8.498.12266644932420958618394322959047940542", "1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098", "1.2.826.0.1.3680043.8.498.33479806419966566464727310013949313521", "1.2.826.0.1.3680043.8.498.26002972602557466222471260313257965551", "1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759", "1.2.826.0.1.3680043.8.498.30672390215843894382866701432377109444", "1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119", "1.2.826.0.1.3680043.8.498.76222770668660274805993655800271653712", "1.2.826.0.1.3680043.8.498.57829173797358025677240411871600928097", "1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943", "1.2.826.0.1.3680043.8.498.29384016102624554484300773136383156153", "1.2.826.0.1.3680043.8.498.56880070152758706555016296832842012883", "1.2.826.0.1.3680043.8.498.12257119852150088212071954292747252243", "1.2.826.0.1.3680043.8.498.67456402757136973484427155324664966595", "1.2.826.0.1.3680043.8.498.12326930847104022196329390608780066947", "1.2.826.0.1.3680043.8.498.62420632149863295140175740643640707423", "1.2.826.0.1.3680043.8.498.58654915518882480245974544469656013058", "1.2.826.0.1.3680043.8.498.13004048880335015585285100185910148808", "1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269", "1.2.826.0.1.3680043.8.498.11912618147674655465071922267060032851", "1.2.826.0.1.3680043.8.498.46745172905542117506863575411125924241", "1.2.826.0.1.3680043.8.498.63195088465290711580668024587233364997", "1.2.826.0.1.3680043.8.498.24866943620580666216571030343221534720", "1.2.826.0.1.3680043.8.498.32328699307307661181881949763360098439", "1.2.826.0.1.3680043.8.498.12408933790879905193765468926022393696", "1.2.826.0.1.3680043.8.498.42965031665568825650795822006244115609", "1.2.826.0.1.3680043.8.498.82591599611540849522609405219902904134", "1.2.826.0.1.3680043.8.498.63683477631564232018930819747238111809", "1.2.826.0.1.3680043.8.498.17836815413824722543229791044312174885", "1.2.826.0.1.3680043.8.498.65124952546314712352837396207287384405", "1.2.826.0.1.3680043.8.498.61490475141024845836646569993549106628", "1.2.826.0.1.3680043.8.498.40608233778918249560056537030108548037", "1.2.826.0.1.3680043.8.498.11716730437469973434747875401501334429", "1.2.826.0.1.3680043.8.498.19830663206351645385520268848711462246", "1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730", "1.2.826.0.1.3680043.8.498.85326542895076582985200613540648336047", "1.2.826.0.1.3680043.8.498.41240503248178304281405515398211762506", "1.2.826.0.1.3680043.8.498.30517381653283520211188050362394799510", "1.2.826.0.1.3680043.8.498.11340429352365972769264064439788186260", "1.2.826.0.1.3680043.8.498.32571577475600416042179474541176580933", "1.2.826.0.1.3680043.8.498.28490861391717560759667589681288427737", "1.2.826.0.1.3680043.8.498.51886877282987244004912794726043055730", "1.2.826.0.1.3680043.8.498.93104685852024334015651558514195715956", "1.2.826.0.1.3680043.8.498.70705483212961922828901596159131697717", "1.2.826.0.1.3680043.8.498.84238814030979264404768702945290704724", "1.2.826.0.1.3680043.8.498.13371001997511542011416510364359588752", "1.2.826.0.1.3680043.8.498.40196271329411009311273016603705560443", "1.2.826.0.1.3680043.8.498.36679002788952511154610176614545192374", "1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611", "1.2.826.0.1.3680043.8.498.77416382267531567401698555072875486512", "1.2.826.0.1.3680043.8.498.71636001534145439773297498405674521629", "1.2.826.0.1.3680043.8.498.63216944317982327305447295456609309952", "1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762", "1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733", "1.2.826.0.1.3680043.8.498.95490888360439852685783476266753304657", "1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290", "1.2.826.0.1.3680043.8.498.12616376042564380071463081913949255574", "1.2.826.0.1.3680043.8.498.15428051518761311558410606976607852351", "1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837", "1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745", "1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310", "1.2.826.0.1.3680043.8.498.96054287426681123062488435620352206601", "1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946", "1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902", "1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186", "1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206", "1.2.826.0.1.3680043.8.498.99578894341571394220300473118855007095", "1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339", "1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156", "1.2.826.0.1.3680043.8.498.11495133447993934376048707763538781434", "1.2.826.0.1.3680043.8.498.11642678020345468499556230871411308014", "1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860", "1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660", "1.2.826.0.1.3680043.8.498.20969353615978236040448556356026349615", "1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924", "1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721", "1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597", "1.2.826.0.1.3680043.8.498.44441813293962885868294317830142850713", "1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479", "1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116", "1.2.826.0.1.3680043.8.498.50424421287869498746932271082980000521", "1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665", "1.2.826.0.1.3680043.8.498.86715997235054599024919893656486735623", "1.2.826.0.1.3680043.8.498.87465202523430400780596885230492167155", "1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055", "1.2.826.0.1.3680043.8.498.94252884278314596961426629318911909075", "1.2.826.0.1.3680043.8.498.95351925765761617117057799846763403963", "1.2.826.0.1.3680043.8.498.11302750434639976191333636496428935632", "1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299", "1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090", "1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314", "1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419", "1.2.826.0.1.3680043.8.498.12638205949145637289277744267473965719", "1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982", "1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922", "1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872", "1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545", "1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619", "1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265", "1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950", "1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103", "1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525", "1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942", "1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291", "1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922", "1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080", "1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258", "1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197", "1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967", "1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855", "1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991", "1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817", "1.2.826.0.1.3680043.8.498.12462498156569316578044317782899399656", "1.2.826.0.1.3680043.8.498.13226961394403558608712498872213808226", "1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314", "1.2.826.0.1.3680043.8.498.13393268272072264469105890769640363680", "1.2.826.0.1.3680043.8.498.17056997047488992455301411194938263817", "1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871", "1.2.826.0.1.3680043.8.498.31898017986762601111404430885432769100", "1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071", "1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853", "1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778", "1.2.826.0.1.3680043.8.498.41769275806689876192405091865340286535", "1.2.826.0.1.3680043.8.498.42051385452979123325023608442860201782", "1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177", "1.2.826.0.1.3680043.8.498.61333047594780344399535845209036589563", "1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756", "1.2.826.0.1.3680043.8.498.69002590469755721401448924002989552840", "1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220", "1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497", "1.2.826.0.1.3680043.8.498.90893882071188409362296099956462540506", "1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916", "1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391", "1.2.826.0.1.3680043.8.498.98039072306618106705944452906395982381", "1.2.826.0.1.3680043.8.498.10181291283712008544475048457589664787", "1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115", "1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884", "1.2.826.0.1.3680043.8.498.10659781998801845781246073928620795082", "1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826", "1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359", "1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699", "1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391", "1.2.826.0.1.3680043.8.498.93301337627536436086612556500687157032", "1.2.826.0.1.3680043.8.498.22938598188283919466040132624303635817", "1.2.826.0.1.3680043.8.498.92285140961160393284537044926797259671", "1.2.826.0.1.3680043.8.498.11906266535122698149967624908585010734", "1.2.826.0.1.3680043.8.498.45462389009834095091353507592651342877", "1.2.826.0.1.3680043.8.498.17909590711893200514107254988453660857", "1.2.826.0.1.3680043.8.498.27311413427137994435573903541974122355", "1.2.826.0.1.3680043.8.498.99178010550062268668346041362629896201", "1.2.826.0.1.3680043.8.498.78188863861812651079751164964756388411", "1.2.826.0.1.3680043.8.498.11610585947047502123824835405421129552", "1.2.826.0.1.3680043.8.498.74206939384612669977629625310277500302", "1.2.826.0.1.3680043.8.498.11654064547593365398103107850228629066", "1.2.826.0.1.3680043.8.498.64946212153370539647301308676689632525", "1.2.826.0.1.3680043.8.498.12965787822216495014894819598440431626", "1.2.826.0.1.3680043.8.498.13087770868099843948960419968576882974", "1.2.826.0.1.3680043.8.498.41057428195711190931490429649014776496", "1.2.826.0.1.3680043.8.498.11911688823908554151914734938172012708", "1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363", "1.2.826.0.1.3680043.8.498.18671006890719350655167441375718348829", "1.2.826.0.1.3680043.8.498.23081294963277525274551516232134221416", "1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331", "1.2.826.0.1.3680043.8.498.10425131723456547056365404016184663035", "1.2.826.0.1.3680043.8.498.13246496699722524712272293738359002501", "1.2.826.0.1.3680043.8.498.12401147012438969595202076588471168894", "1.2.826.0.1.3680043.8.498.89326347774248795083360112232993521530", "1.2.826.0.1.3680043.8.498.66960891947495601371173426277430301127", "1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695", "1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785", "1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066", "1.2.826.0.1.3680043.8.498.12499208731138180053966892599139021038", "1.2.826.0.1.3680043.8.498.81822709772232914298414700148454511661", "1.2.826.0.1.3680043.8.498.21135201439483157418374033420156525658", "1.2.826.0.1.3680043.8.498.53649895712172408476570770924813501118", "1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669", "1.2.826.0.1.3680043.8.498.12800130397185143653180509040746675358", "1.2.826.0.1.3680043.8.498.74214829456846525506232367308573550186", "1.2.826.0.1.3680043.8.498.11641274367861998748475337224297076397", "1.2.826.0.1.3680043.8.498.44863396093454506630473164990623458425", "1.2.826.0.1.3680043.8.498.63126604639589769235241216875934773992", "1.2.826.0.1.3680043.8.498.11935893582662912138522962760478513770", "1.2.826.0.1.3680043.8.498.19197037365998103318720742154440921358", "1.2.826.0.1.3680043.8.498.44670636848627131733157841382853275817", "1.2.826.0.1.3680043.8.498.19813267030974842239752491935957206955", "1.2.826.0.1.3680043.8.498.13203246712933982189680087013566455075", "1.2.826.0.1.3680043.8.498.11577630811395858629711985689258647804", "1.2.826.0.1.3680043.8.498.83602205818477323486169938182117122488", "1.2.826.0.1.3680043.8.498.48367523758673249284754003765836877239", "1.2.826.0.1.3680043.8.498.10459044164648238749799867924228983674", "1.2.826.0.1.3680043.8.498.43870319732657730742794291443940225011", "1.2.826.0.1.3680043.8.498.73254365064404586162477089636000427751", "1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343", "1.2.826.0.1.3680043.8.498.19178214677492694135853596920291058397", "1.2.826.0.1.3680043.8.498.99155175553749543768821558247962844052", "1.2.826.0.1.3680043.8.498.80671597537348112977780954910198682272", "1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801", "1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706", "1.2.826.0.1.3680043.8.498.66031142975529812579758652269360365398", "1.2.826.0.1.3680043.8.498.43300763182450509440662392516941842389", "1.2.826.0.1.3680043.8.498.31806829168516955528191705338336428721", "1.2.826.0.1.3680043.8.498.24283484945988825507450239904606754405", "1.2.826.0.1.3680043.8.498.13078009378379325641560321462188164769", "1.2.826.0.1.3680043.8.498.94027547274004055924993074754507802303", "1.2.826.0.1.3680043.8.498.19821498229702528191367700655079805824", "1.2.826.0.1.3680043.8.498.28628472249424790112957421215774036156", "1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650", "1.2.826.0.1.3680043.8.498.13363562142644508091380459785810556567", "1.2.826.0.1.3680043.8.498.34895742092137516664772577251243433579", "1.2.826.0.1.3680043.8.498.58685941646495891824559391997783302656", "1.2.826.0.1.3680043.8.498.54316764984549863227549326352573342232", "1.2.826.0.1.3680043.8.498.95864678405973312042815189771902289608", "1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270", "1.2.826.0.1.3680043.8.498.24432243084670627958434695475653885175", "1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028", "1.2.826.0.1.3680043.8.498.11052656375391331522834364601050945860", "1.2.826.0.1.3680043.8.498.10407865232593182855530878478701744409", "1.2.826.0.1.3680043.8.498.63105151375349194807502162946823966410", "1.2.826.0.1.3680043.8.498.69131433601281694165384475765166382765", "1.2.826.0.1.3680043.8.498.11448326350892044215419380818679679579", "1.2.826.0.1.3680043.8.498.26325951519908477683595799515094966069", "1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265", "1.2.826.0.1.3680043.8.498.94594009911808653224418094594012527944", "1.2.826.0.1.3680043.8.498.56146582420561901778012523946835033628", "1.2.826.0.1.3680043.8.498.34040631818813472592501398076075999835"]
tr = pd.read_csv(DATA / 'train.csv', dtype={'StudyInstanceUID': str})
L = V.TARGETS
gold = tr[tr[L].notna().all(axis=1)]['StudyInstanceUID'].tolist()
assert len(gold) == 58, len(gold)
uids = gold + [u for u in REFB if u not in set(gold)]
ser = pd.read_csv(DATA / 'train_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
by = {u: g for u, g in ser.groupby('StudyInstanceUID', sort=False)}


In [ ]:
# fix infer.py: for series with > N_SLICES files it calls pydicom.dcmread(ds.filename), which raises TypeError with this
# pydicom, so that view was silently left empty. Keep the file path on each header and read pixels from it.
import inspect, pydicom
_src = inspect.getsource(V._load_source_slices)
_src = _src.replace("pydicom.dcmread(path, stop_before_pixels=True) for path in files", "_hdr(path) for path in files")
_src = _src.replace("pydicom.dcmread(ds.filename)", "pydicom.dcmread(ds._vpath)")
assert '_hdr(path)' in _src and 'ds._vpath' in _src
def _hdr(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True); ds._vpath = path; return ds
V._hdr = _hdr
exec(_src, V.__dict__)
_t = sorted(glob.glob(str(DATA / 'train_series' / gold[0] / '*')))
print('fix check', [None if s is None else s.shape for s in (V._load_source_slices(Path(p)) for p in _t[:3])], flush=True)


In [ ]:
NERR = [0]
# same as infer.load_study_tensor, but reads train_series; fp16 autocast (T4 has no bf16)
def load_train(uid):
    rows = by.get(uid)
    views = np.zeros((5, V.N_SLICES, V.RES, V.RES), dtype=np.uint8); present = np.zeros(5, dtype=bool)
    if rows is not None:
        for vi, (plane, fluid) in enumerate(V.VIEWS5):
            for row in rows[(rows['Anatomical_Plane'] == plane) & (rows['Fluid_Sensitive'] == fluid)].itertuples(index=False):
                try: st = V._load_source_slices(DATA / 'train_series' / uid / str(row.SeriesInstanceUID))
                except Exception as e:
                    st = None; NERR[0] += 1
                if st is not None:
                    views[vi] = st; present[vi] = bool(np.any(st)); break
    if not present.any(): present[0] = True
    pos = np.rint(np.linspace(0, V.N_SLICES - 1, V.FRAMES)).astype(int)
    return views[:, pos], present
cks = sorted(CK.glob('f*_best.pt'))
models = [V.load_model(c) for c in cks]
print('folds', [c.name for c in cks], flush=True)
P = {k: [] for k in range(len(models))}
t0 = time.time()
with torch.inference_mode():
    for i, u in enumerate(uids):
        g, pr = load_train(u)
        x = torch.from_numpy(g).unsqueeze(0).cuda(); p = torch.from_numpy(pr).unsqueeze(0).cuda()
        with torch.autocast('cuda', dtype=torch.float16):
            for k, m in enumerate(models):
                P[k].append(m(x, p).float().squeeze(0).cpu().numpy())
        if (i + 1) % 20 == 0: print(i + 1, len(uids), '%.0fs' % (time.time() - t0), flush=True)
for k in P:
    pd.DataFrame(P[k], columns=L, index=uids).rename_axis('StudyInstanceUID').to_csv('/kaggle/working/vjepa_f%d_logits.csv' % k)
print('view load errors', NERR[0], flush=True)


In [ ]:
from sklearn.metrics import roc_auc_score
G = tr.set_index('StudyInstanceUID').loc[gold, L].astype(int)
mean = sum(pd.read_csv('/kaggle/working/vjepa_f%d_logits.csv' % k, index_col=0) for k in P) / len(P)
res = {}
for k in list(P) + ['mean']:
    d = mean if k == 'mean' else pd.read_csv('/kaggle/working/vjepa_f%d_logits.csv' % k, index_col=0)
    a = {l: roc_auc_score(G[l], d.loc[gold, l]) for l in L}; res[str(k)] = a
    print(k, 'gold macro %.4f' % np.mean(list(a.values())), ' '.join('%.3f' % a[l] for l in L))
json.dump(res, open('/kaggle/working/vjepa_gold_auc.json', 'w'), indent=1)
